# DITTO round 2 on Colab: tuned SFT, gentle DITTO, honest selection on dev

Runs top to bottom with **Runtime > Run all** on an **A100** (or L4). Everything is saved to
`MyDrive/aphasia_ditto_colab_r2/` as it goes; after a disconnect, run all again and finished work is reused.

Fixed since round 1 (evidence in the repo commits and tests):
- **Prompt format v2**: every option is labelled in words, with levels relative to the current picture
  ("one level easier") and hint names that match the error names ("meaning error" / "meaning hint").
- **Data v4**: options follow the contract (a hint already shown is never offered again); 161 training
  situations, including more rule-G cases near the 20-minute boundary. Dev 33, test 55, test_v1 10.
- **No SFT early stop**: the number of SFT steps is chosen on dev.
- **Gentle DITTO**: lower learning rate, higher beta, an RPO term (NLL of the demonstrated action), and dev
  agreement after every round; a DITTO round is kept only if it beats its own SFT on dev.
- **prompted_all** scores long prompts one option at a time, so it no longer runs out of memory.

Qwen3-1.7B with option scoring throughout (the model picks the most likely valid option).

Plan:
1. GPU, Drive, install (with the torchao fix and a LoRA self-check).
2. Tune on dev only: SFT steps (60 / 120 / 200), then 3 gentle-DITTO settings on top of the best SFT.
3. Choose the delivered model among all candidates by dev agreement ("SFT only" is a candidate) and save it.
4. 3 seeds of the chosen pipeline; the delivered seed is chosen on dev and saved.
5. Test (55) and test_v1 (10), exactly once: base, prompted, prompted_all, SFT, DITTO and bandit, with
   95% CIs, paired differences, per-rule accuracy, mean and range over seeds.
6. Zip the best adapter and the tables.

The demonstrations are fictional, generated from demonstrator A's written strategy, pending human review.

In [ ]:
# 1. GPU check, dtype and rough time estimates
import subprocess, sys, time
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU (A100 or L4), then run all again."
GPU = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
major, _ = torch.cuda.get_device_capability(0)
if major >= 8 and torch.cuda.is_bf16_supported():
    DTYPE = "bfloat16"
else:
    # No native bfloat16 (T4 and older): float32 is safe for LoRA on a 1.7B model in 16 GB, but slower.
    DTYPE = "float32"
    print("WARNING: no native bfloat16 on this GPU. Using float32: safe but several times slower. Prefer A100 or L4.")
SPEED = 1.0 if "A100" in GPU or "H100" in GPU else 0.5 if "L4" in GPU else 0.2 if "T4" in GPU else 0.4
print(f"GPU: {GPU}, {GPU_GB:.0f} GB, dtype {DTYPE}")
# Rough minutes on an A100 (round 1 measured 1.1 min per option-scoring run; not measured for round 2).
A100_MIN = {"setup (clone, install, model download)": 6, "tuning: 3 SFT-only runs (+ dev eval)": 8,
            "tuning: 3 gentle-DITTO runs (+ dev eval)": 12, "seeds 1 and 2 (+ dev eval)": 8,
            "test + test_v1 (once; prompted_all prompts are ~32k tokens)": 10, "zip": 1}
for stage, m in A100_MIN.items():
    print(f"  {stage:62s} ~{m / SPEED:4.0f} min")
print(f"  {'total':62s} ~{sum(A100_MIN.values()) / SPEED:4.0f} min")

In [ ]:
# 2. Google Drive: everything persistent lives here (a new folder: nothing from round 1 is reused)
from google.colab import drive
drive.mount("/content/drive")
import json, os, shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/aphasia_ditto_colab_r2")
RUNS, EVAL, CONFIGS, LOGS, RESULTS = (DRIVE / d for d in ("runs", "eval", "configs", "logs", "results"))
for d in (RUNS, EVAL, CONFIGS, LOGS, RESULTS):
    d.mkdir(parents=True, exist_ok=True)
print("saving to", DRIVE)
print("already there:", sorted(p.name for p in RUNS.iterdir()) or "nothing (fresh start)")

In [ ]:
# 3. Clone the branch and install
REPO_URL, BRANCH, REPO = "https://github.com/burakdon/HackathonPOPO.git", "feature/ditto-training", Path("/content/HackathonPOPO")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO)
# An editable install is only picked up by a fresh kernel; the repo on sys.path makes it importable now.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
# The project pins Python 3.11 for uv; Colab runs 3.12, which the code supports, hence --ignore-requires-python.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python", "-e", ".[llm]"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip install failed (see above)"
# Colab preinstalls torchao 0.10, which peft rejects ("only versions above 0.16.0 are supported");
# training does not use it, so remove it before peft is first imported.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
LORA_CHECK = '''
import peft, torch
from peft import LoraConfig, get_peft_model
from transformers import Qwen3Config, Qwen3ForCausalLM
m = Qwen3ForCausalLM(Qwen3Config(vocab_size=128, hidden_size=32, intermediate_size=64, num_hidden_layers=1,
                                 num_attention_heads=2, num_key_value_heads=1, head_dim=16))
m = get_peft_model(m, LoraConfig(r=4, lora_alpha=8, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
loss = m(input_ids=torch.tensor([[1, 2, 3]]), labels=torch.tensor([[1, 2, 3]])).loss
loss.backward()
print("LORA_OK peft", peft.__version__, "trainable", sum(p.numel() for p in m.parameters() if p.requires_grad))
'''
chk = subprocess.run([sys.executable, "-c", LORA_CHECK], capture_output=True, text=True)
print(chk.stdout.strip(), chk.stderr[-2000:] if chk.returncode else "")
assert "LORA_OK" in chk.stdout, "peft / LoRA self-check failed (see above); do not start training"
import transformers, peft, accelerate
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| accelerate", accelerate.__version__)

info_path = DRIVE / "run_info.json"
info = json.loads(info_path.read_text()) if info_path.exists() else {"sessions": []}
if info["sessions"] and info["sessions"][0]["commit"] != COMMIT:
    print(f"WARNING: earlier runs in Drive used commit {info['sessions'][0]['commit'][:8]}, now {COMMIT[:8]}.")
info["sessions"].append({"commit": COMMIT, "gpu": GPU, "dtype": DTYPE, "started": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__})
info_path.write_text(json.dumps(info, indent=1))

from huggingface_hub import model_info
BASE = ("Qwen/Qwen3-1.7B", "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e")
try:
    model_info(BASE[0], revision=BASE[1])
    print("base model reachable without login")
except Exception as e:
    print("base model not reachable anonymously:", type(e).__name__, "- logging in")
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(userdata.get("HF_TOKEN"))          # Colab secret named HF_TOKEN
    except Exception:
        from huggingface_hub import notebook_login
        notebook_login()

## Helpers: configs, resumable training and evaluation
One practice config (Qwen3-1.7B, option scoring, prompt format v2, prompted_all on the v4 training file,
Drive directories). Each candidate gets a DITTO config: repo `configs/ditto_llm.yaml` plus its changes, v4
data, no SFT early stop and selection on dev. A run counts as finished only if its manifest says
`completed`; partial runs are deleted and redone. An evaluation is reused if its JSON exists in Drive.

In [ ]:
import yaml

DEMOS = "data/demonstrations/demonstrator_a_v4.jsonl"
SFT_STEPS = [60, 120, 200]
GENTLE = {   # DITTO on top of the chosen SFT, 3 rounds each; rounds are kept only if they beat SFT on dev
    "g1": {"learning_rate": 1.0e-5, "beta": 0.3, "rpo_alpha": 1.0, "steps_per_round": 15},
    "g2": {"learning_rate": 2.0e-5, "beta": 0.2, "rpo_alpha": 0.5, "steps_per_round": 30},
    "g3": {"learning_rate": 5.0e-6, "beta": 0.5, "rpo_alpha": 1.0, "steps_per_round": 30},
}


def practice_path() -> Path:
    p = yaml.safe_load((REPO / "configs/practice.yaml").read_text())
    p["llm"].update(base_model=BASE[0], revision=BASE[1], dtype=DTYPE, decoding="score", prompt_format="v2",
                    prompted_all_demos=DEMOS, runs_dir=str(RUNS), timeout_s=300)
    p["eval_dir"] = str(EVAL)
    out = CONFIGS / "practice_r2.yaml"
    out.write_text(yaml.safe_dump(p, sort_keys=False))
    return out


def ditto_path(name: str, sft_steps: int, gentle: str | None, seed: int) -> Path:
    d = yaml.safe_load((REPO / "configs/ditto_llm.yaml").read_text())
    d.update(practice_config=str(practice_path()), demonstrations=DEMOS, seed=seed, select_on="dev")
    d["sft"].update(max_steps=sft_steps, stop_loss=0.0)
    if gentle is None:
        d["ditto"]["rounds"] = 0                      # SFT only
    else:
        d["ditto"].update(GENTLE[gentle])
    out = CONFIGS / f"ditto_{name}_s{seed}.yaml"
    out.write_text(yaml.safe_dump(d, sort_keys=False))
    return out


def run_id(name: str, seed: int) -> str:
    return f"{name}-s{seed}"


def stream(cmd: list[str], log: Path) -> None:
    # Run a module, echo its output (minus progress bars) and keep a full log in Drive.
    with open(log, "a") as f, subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                               text=True, bufsize=1) as p:
        for line in p.stdout:
            f.write(line)
            if "it/s]" not in line and "s/it]" not in line and "Warning" not in line:
                print(line, end="")
    if p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}); log: {log}")


def manifest(rid: str) -> dict | None:
    p = RUNS / rid / "manifest.json"
    return json.loads(p.read_text()) if p.exists() else None


def train(name: str, sft_steps: int, gentle: str | None, seed: int) -> dict:
    rid = run_id(name, seed)
    m = manifest(rid)
    if m and m.get("status") == "completed":
        print(f"[skip] {rid} already trained ({m['seconds']} s)")
        return m
    if (RUNS / rid).exists():
        print(f"[redo] {rid} was incomplete (status {m and m.get('status')}); deleting it")
        shutil.rmtree(RUNS / rid)
    what = f"SFT {sft_steps} steps" + (f" + gentle DITTO {gentle} {GENTLE[gentle]}" if gentle else " only")
    print(f"[train] {rid}: {what}")
    t0 = time.time()
    stream([sys.executable, "-m", "training.ditto", "--config", str(ditto_path(name, sft_steps, gentle, seed)),
            "--run-id", rid], LOGS / f"train_{rid}.log")
    print(f"[done] {rid} in {(time.time() - t0) / 60:.1f} min")
    return manifest(rid)


def evaluate(rid: str, split: str, conditions: list[str]) -> dict:
    out = EVAL / f"{rid}-{split}.json"
    if out.exists():
        print(f"[skip] {split} evaluation of {rid} already in Drive")
        return json.loads(out.read_text())
    stream([sys.executable, "-m", "evaluation.compare", "--run", rid, "--split", split, "--practice",
            str(practice_path()), "--conditions", *conditions], LOGS / f"eval_{rid}_{split}.log")
    return json.loads(out.read_text())


def best_round(m: dict) -> tuple[str | None, int | None]:
    # Best DITTO round (>= 1) on dev; ties go to the earlier round. None for SFT-only runs.
    rounds = {k: v for k, v in m.get("selection", {}).get("agreement", {}).items() if k != "sft"}
    if not rounds:
        return None, None
    order = list(rounds)
    k = max(order, key=lambda r: (rounds[r], -order.index(r)))
    return k, rounds[k]


STAGE_TIMES = {}
from evaluation.compare import bootstrap_ci, fmt_agree, to_markdown
import pandas as pd

## 4a. Tuning, dev only: number of SFT steps
Three SFT-only runs (no early stop). Dev agreement is measured inside the trainer after SFT and again
with `evaluation.compare --split dev` (which also gives the 95% interval). Nothing here reads test.

In [ ]:
t_stage = time.time()
sft_rows = []
for steps in SFT_STEPS:
    name = f"sft{steps}"
    m = train(name, steps, None, 0)
    rep = evaluate(run_id(name, 0), "dev", [f"sft:{run_id(name, 0)}"])
    r = rep["results"][f"sft:{run_id(name, 0)}"]
    sft_rows.append({"candidate": f"SFT only, {steps} steps", "name": name, "steps": steps,
                     "dev": r["agree_end_to_end"], "n": r["n"],
                     "dev (95% CI)": fmt_agree(r["agree_end_to_end"], r["n"], r["agree_end_to_end_ci95"]),
                     "train min": round(m["seconds"] / 60, 1)})
STAGE_TIMES["tuning_sft"] = time.time() - t_stage
sft_df = pd.DataFrame(sft_rows)
BEST_STEPS = sorted(sft_rows, key=lambda r: (-r["dev"], r["steps"]))[0]["steps"]   # ties: fewer steps
print("best SFT steps on dev:", BEST_STEPS)
sft_df[["candidate", "dev (95% CI)", "train min"]]

## 4b. Tuning, dev only: gentle DITTO on top of the best SFT
Each setting trains SFT for the chosen number of steps and then 3 DITTO rounds, measuring dev agreement
after every round. A setting's dev score is its best round (round >= 1).

In [ ]:
t_stage = time.time()
ditto_rows = []
for g in GENTLE:
    name = f"sft{BEST_STEPS}-{g}"
    m = train(name, BEST_STEPS, g, 0)
    k, score = best_round(m)
    rid = run_id(name, 0)
    rep = evaluate(rid, "dev", [f"sft:{rid}", f"ditto:{rid}@{k}"])
    r, s = rep["results"][f"ditto:{rid}@{k}"], rep["results"][f"sft:{rid}"]
    if r["agree_end_to_end"] != score:      # bfloat16 rounding can flip a near-tie between the two passes
        print(f"note: dev agreement {score} inside the trainer vs {r['agree_end_to_end']} from the evaluator")
    ditto_rows.append({"candidate": f"SFT {BEST_STEPS} + gentle DITTO {g}, {k}", "name": name, "setting": g,
                       "round": k, "dev": score, "n": r["n"], "sft dev": s["agree_end_to_end"],
                       "dev (95% CI)": fmt_agree(score, r["n"], r["agree_end_to_end_ci95"]),
                       "per stage": m["selection"]["agreement"], "train min": round(m["seconds"] / 60, 1),
                       "settings": GENTLE[g]})
STAGE_TIMES["tuning_ditto"] = time.time() - t_stage
ditto_df = pd.DataFrame(ditto_rows)
ditto_df[["candidate", "dev (95% CI)", "sft dev", "per stage", "train min"]]

## 5. Choose the delivered model (dev agreement; ties: SFT only, then fewer steps / earlier round)
"SFT only" is an explicit candidate. The choice is saved and reused on resume, so it cannot change after
the test evaluation.

In [ ]:
choice_path = DRIVE / "choice.json"
if choice_path.exists():
    choice = json.loads(choice_path.read_text())
    print("[reuse] choice made earlier:", choice["delivered"])
else:
    best_sft = sorted(sft_rows, key=lambda r: (-r["dev"], r["steps"]))[0]
    best_ditto = sorted(ditto_rows, key=lambda r: (-r["dev"], list(GENTLE).index(r["setting"])))[0]
    use_ditto = best_ditto["dev"] > best_sft["dev"]
    n = best_sft["n"]
    if use_ditto:
        delivered = f"DITTO (SFT {BEST_STEPS} steps + gentle setting {best_ditto['setting']}, best dev round per seed)"
        why = (f"gentle DITTO {best_ditto['setting']} reaches {best_ditto['dev']}/{n} on dev at {best_ditto['round']}, "
               f"above the best SFT-only candidate ({best_sft['dev']}/{n} at {best_sft['steps']} steps)")
    else:
        delivered = f"SFT only ({best_sft['steps']} steps)"
        why = (f"no gentle-DITTO setting beats SFT only on dev (best DITTO {best_ditto['dev']}/{n}, "
               f"SFT {best_sft['dev']}/{n}); the delivered model is named SFT, not DITTO")
    choice = {"delivered": delivered, "kind": "ditto" if use_ditto else "sft", "sft_steps": BEST_STEPS,
              "gentle": best_ditto["setting"], "why": why, "sft_candidates": sft_rows,
              "ditto_candidates": [{k: v for k, v in r.items()} for r in ditto_rows]}
    choice_path.write_text(json.dumps(choice, indent=1, default=str))
PIPE = f"sft{choice['sft_steps']}-{choice['gentle']}"     # seeds always train SFT + DITTO, so both columns exist
print("DELIVERED:", choice["delivered"])
print("WHY:", choice["why"])
print("seed pipeline:", PIPE)

## 6. Seeds 1 and 2
Every seed trains the chosen SFT steps plus the chosen gentle-DITTO setting, so both the SFT and the
DITTO columns exist. Per seed, the DITTO adapter is its best round on dev. The delivered seed is chosen on
dev (ties: lowest seed) and saved before any test evaluation.

In [ ]:
t_stage = time.time()
SEEDS = [0, 1, 2]
seed_info = {}
for seed in SEEDS:
    m = train(PIPE, choice["sft_steps"], choice["gentle"], seed)
    k, score = best_round(m)
    rid = run_id(PIPE, seed)
    stage = k if choice["kind"] == "ditto" else "sft"
    dev_score = score if choice["kind"] == "ditto" else m["selection"]["agreement"]["sft"]
    seed_info[seed] = {"run": rid, "ditto_round": k, "ditto_dev": score, "sft_dev": m["selection"]["agreement"]["sft"],
                       "delivered_stage": stage, "delivered_dev": dev_score}
STAGE_TIMES["seeds"] = time.time() - t_stage
seed_path = DRIVE / "seed_choice.json"
if seed_path.exists():
    BEST_SEED = json.loads(seed_path.read_text())["best_seed"]
    print("[reuse] delivered seed chosen earlier:", BEST_SEED)
else:
    BEST_SEED = sorted(SEEDS, key=lambda s: (-seed_info[s]["delivered_dev"], s))[0]
    seed_path.write_text(json.dumps({"seed_info": seed_info, "best_seed": BEST_SEED}, indent=1))
display(pd.DataFrame(seed_info).T)
print("delivered seed:", BEST_SEED, seed_info[BEST_SEED])

## 7. Test evaluation: exactly once (test 55 and test_v1 10)
Seed 0's run is evaluated with every condition (base, prompted, prompted_all and bandit do not depend on
the seed), seeds 1 and 2 with SFT and DITTO. "DITTO" is each seed's best DITTO round on dev. Intervals:
95% percentile bootstrap over situations; for SFT and DITTO each situation's agreement is first averaged
over the 3 seeds. Results are stored in Drive and reloaded on resume.

In [ ]:
import numpy as np
from statistics import mean

assert (DRIVE / "choice.json").exists() and (DRIVE / "seed_choice.json").exists(), "choose model and seed first"
TEST_SPLITS = ["test", "test_v1"]
t_stage = time.time()
reports = {}
for split in TEST_SPLITS:
    reports[split] = {}
    for seed in SEEDS:
        rid, k = seed_info[seed]["run"], seed_info[seed]["ditto_round"]
        conds = [f"sft:{rid}", f"ditto:{rid}@{k}"]
        if seed == 0:
            conds = ["base", "prompted", "prompted_all"] + conds + ["bandit"]
        reports[split][seed] = evaluate(rid, split, conds)
STAGE_TIMES["test"] = time.time() - t_stage

SFT_C = {s: f"sft:{seed_info[s]['run']}" for s in SEEDS}
DIT_C = {s: f"ditto:{seed_info[s]['run']}@{seed_info[s]['ditto_round']}" for s in SEEDS}
GROUPS = {"base": ["base"], "prompted": ["prompted"], "prompted_all": ["prompted_all"],
          "SFT": list(SFT_C.values()), "DITTO": list(DIT_C.values()), "bandit": ["bandit"]}
DELIVERED = "DITTO" if choice["kind"] == "ditto" else "SFT"


def analyse(split: str) -> tuple[list[str], dict]:
    test = reports[split]
    assert all(test[s]["example_ids"] == test[0]["example_ids"] for s in SEEDS)
    res = {c: test[0]["results"][c] for c in ("base", "prompted", "prompted_all")}
    for s in SEEDS:
        res[SFT_C[s]] = test[s]["results"][SFT_C[s]]
    for s in SEEDS:
        res[DIT_C[s]] = test[s]["results"][DIT_C[s]]
    res["bandit"] = test[0]["results"]["bandit"]
    combined = {**test[0], "run_id": f"{PIPE} (seeds {', '.join(map(str, SEEDS))})", "results": res}
    n = res["base"]["n"]
    hits = {g: np.mean([[p["agree_end_to_end"] for p in res[c]["per_example"]] for c in conds], axis=0)
            for g, conds in GROUPS.items()}
    summary = []
    for g, conds in GROUPS.items():
        ag = [res[c]["agree_end_to_end"] for c in conds]
        va = [res[c]["valid_output"] for c in conds if res[c]["valid_output"] is not None]
        fb = [res[c]["fallbacks"] for c in conds]
        shots = res[conds[0]].get("model_detail", {}).get("few_shot_examples")
        label = g + (f" ({shots} demos)" if g.startswith("prompted") and shots else "") + (" [delivered]" if g == DELIVERED else "")
        summary.append({"model": label, "runs": len(conds), "agree": fmt_agree(round(mean(ag), 2), n, bootstrap_ci(hits[g])),
                        "range over seeds": f"{min(ag)}-{max(ag)}" if len(conds) > 1 else "",
                        "valid output": f"{mean(va):.1f}/{n}" if va else "n/a", "fallbacks": f"{mean(fb):.1f}",
                        "median latency ms": round(mean(res[c]["latency_ms_median"] for c in conds))})
    idx = np.random.default_rng(0).integers(0, n, (2000, n))
    paired = []
    for g in ("base", "prompted", "prompted_all", "SFT", "bandit"):
        diff = hits["DITTO"] - hits[g]
        boots = diff[idx].mean(axis=1)
        paired.append({"comparison": f"DITTO - {g}", "difference": round(100 * diff.mean(), 1),
                       "95% CI": f"{100 * np.quantile(boots, 0.025):.1f} to {100 * np.quantile(boots, 0.975):.1f}"})
    rules = sorted({p["rule"] for p in res["base"]["per_example"]})
    rule_of = [p["rule"] for p in res["base"]["per_example"]]
    per_rule = {}
    for g in GROUPS:
        for rule in rules:
            h = np.array([x for x, r in zip(hits[g], rule_of) if r == rule])
            per_rule.setdefault(rule, {})[g] = fmt_agree(round(float(h.sum()), 2), len(h), bootstrap_ci(h))
    lines = [f"## {split} ({n} situations, evaluated once)", "",
             "| model | runs | agrees end to end (mean over seeds, 95% CI) | range over seeds | valid output | fallbacks | median latency |",
             "|---|---|---|---|---|---|---|"]
    lines += [f"| {r['model']} | {r['runs']} | {r['agree']} | {r['range over seeds']} | {r['valid output']} | "
              f"{r['fallbacks']} | {r['median latency ms']} ms |" for r in summary]
    lines += ["", "Paired differences on the same situations (SFT and DITTO averaged over seeds):", "",
              "| comparison | difference (percentage points) | 95% CI |", "|---|---|---|"]
    lines += [f"| {r['comparison']} | {r['difference']:+.1f} | {r['95% CI']} |" for r in paired]
    lines += ["", f"Per-rule agreement on {split} (SFT and DITTO: mean over seeds per situation):", "",
              "| rule | " + " | ".join(GROUPS) + " |", "|---|" + "---|" * len(GROUPS)]
    lines += [f"| {rule} | " + " | ".join(per_rule[rule][g] for g in GROUPS) + " |" for rule in rules]
    lines += ["", f"Full table ({split}):", "", to_markdown(combined)]
    return lines, {"summary": summary, "paired": paired, "per_rule": per_rule}


lines = [f"# DITTO round 2 on Colab: delivered model `{choice['delivered']}`", "",
         f"GPU {GPU}, dtype {DTYPE}, commit `{COMMIT[:8]}`. Qwen3-1.7B, option scoring, prompt format v2. Training: "
         "161 situations (v4) generated from demonstrator A's written strategy, pending human review. prompted = "
         "the 20 v1 training demonstrations in the prompt; prompted_all = as many v4 training demonstrations as fit "
         "the context window.", "", "## Tuning on dev (33 situations; test never read)", "",
         "| candidate | dev agreement (95% CI) | train min |", "|---|---|---|"]
lines += [f"| {r['candidate']} | {r['dev (95% CI)']} | {r['train min']} |" for r in choice["sft_candidates"]]
lines += [f"| {r['candidate']} (its SFT stage: {r['sft dev']}/{r['n']}; per stage {r['per stage']}) | {r['dev (95% CI)']} | "
          f"{r['train min']} |" for r in choice["ditto_candidates"]]
lines += ["", f"Delivered: **{choice['delivered']}**. Why: {choice['why']}.",
          f"Seeds (dev agreement per stage): {json.dumps(seed_info)}. Delivered seed (chosen on dev): {BEST_SEED}.", ""]
analysis = {}
for split in TEST_SPLITS:
    part, analysis[split] = analyse(split)
    lines += part + [""]
lines += ["Option scoring: every output is a valid option by construction.",
          "Mock preferences of a fictional demonstrator: an exploratory check, not a statistical or clinical validation."]
(RESULTS / "ditto_colab_r2_test.md").write_text("\n".join(lines) + "\n")
(RESULTS / "ditto_colab_r2_summary.json").write_text(json.dumps(
    {"delivered": choice["delivered"], "why": choice["why"], "best_seed": BEST_SEED, "seed_info": seed_info,
     "analysis": analysis, "gpu": GPU, "dtype": DTYPE, "commit": COMMIT,
     "stage_minutes": {k: round(v / 60, 1) for k, v in STAGE_TIMES.items()}}, indent=1, default=str))
for split in TEST_SPLITS:
    print(f"\n=== {split}")
    display(pd.DataFrame(analysis[split]["summary"]))
    display(pd.DataFrame(analysis[split]["paired"]))
    display(pd.DataFrame(analysis[split]["per_rule"]).T)
print(f"report: {RESULTS / 'ditto_colab_r2_test.md'}")

## 8. Bundle: delivered adapter, all runs and tables
`best_adapter/<run_id>/` has the layout of `models/ditto_llm/<run_id>/` in the repo: the delivered stage's
adapter, the manifest, and the dev, test and test_v1 evaluations.

In [ ]:
best = seed_info[BEST_SEED]
best_rid, stage = best["run"], best["delivered_stage"]
best_dir = DRIVE / "best_adapter" / best_rid
best_dir.mkdir(parents=True, exist_ok=True)
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(RUNS / best_rid / stage / f, best_dir / f)
m = manifest(best_rid)
m["delivered_stage"] = stage
(best_dir / "manifest.json").write_text(json.dumps(m, indent=1))
for split in ("test", "test_v1"):
    shutil.copy(EVAL / f"{best_rid}-{split}.json", best_dir / f"eval_{split}.json")

import zipfile


def make_zip(name: str, parts: list[Path]) -> Path:
    # Built outside the folder being zipped, then copied to Drive.
    tmp = (Path("/content") if Path("/content").exists() else Path(__import__("tempfile").gettempdir())) / name
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for part in parts:
            for p in ([part] if part.is_file() else sorted(part.rglob("*"))):
                if p.is_file():
                    z.write(p, p.relative_to(DRIVE))
    shutil.copy(tmp, DRIVE / name)
    print(f"{DRIVE / name}: {tmp.stat().st_size / 1e6:.0f} MB")
    return tmp


meta = [DRIVE / f for f in ("choice.json", "seed_choice.json", "run_info.json")]
full = make_zip("ditto_colab_r2_bundle.zip", [RUNS, EVAL, RESULTS, CONFIGS, LOGS, DRIVE / "best_adapter", *meta])
small = make_zip("ditto_colab_r2_best_and_results.zip", [best_dir, RESULTS, CONFIGS, *meta])
print("stage minutes:", {k: round(v / 60, 1) for k, v in STAGE_TIMES.items()})
from google.colab import files
files.download(str(small))   # delivered adapter + tables; the full bundle (all runs) stays in Drive